# Lab 0：金融数据源体检

## Goal

本 Notebook 对 https://akshare.akfamily.xyz/data/stock/stock.html 进行体检 收敛成一个诊断入口：

1. 分行情层、基础数据层、研报层、新闻与公告层；
2. 对新 Lab 1 依赖的交易所清单、腾讯/新浪行情、通达信 F10 和分红主备源执行连续体检；
3. 记录真实上游、当前状态、耗时、错误与路由决策；
4. 明确东方财富部分接口不可用、`mootdx.bars` 暂缓，不让它们进入自动降级链。

状态只代表本次运行环境，不是长期服务承诺。AKShare 是封装层，真实风险面仍按
腾讯、新浪、东方财富、交易所和通达信分别登记。


## Setup

运行数据写入 `labs/data/lab0/<run_id>/`，稳定汇总另写
`labs/data/lab0/source_healthcheck.csv`。接口只在相应域名调用期间临时直连，
不会改动系统代理。


In [ ]:
import sys
from pathlib import Path

import akshare as ak
import pandas as pd
from IPython.display import display

WORKING_DIR = Path.cwd().resolve()

# 从当前目录及其父级查找 labs/ 目录（其下需有 pyproject.toml）。
# 支持三种启动位置：仓库根目录、labs/、labs/<子目录>/（如 labs/01_银行股定投回测/）。
LABS_DIR = None
for _candidate in [WORKING_DIR, *WORKING_DIR.parents]:
    if _candidate.name == "labs" and (_candidate / "pyproject.toml").is_file():
        LABS_DIR = _candidate
        break
    if (_candidate / "labs" / "pyproject.toml").is_file():
        LABS_DIR = _candidate / "labs"
        break

if LABS_DIR is None:
    raise FileNotFoundError(
        "未找到 labs/pyproject.toml。请从仓库根目录、labs/ 或 labs/ 子目录启动 Jupyter。"
        f" 当前目录：{WORKING_DIR}"
    )

if str(LABS_DIR) not in sys.path:
    sys.path.insert(0, str(LABS_DIR))
sys.path.insert(0, str(LABS_DIR / "01_银行股定投回测"))

from bank_dca import (
    fetch_sina_history,
    fetch_tencent_history,
    flatten_f10,
    tdx_client,
)
from data_source_registry import (
    InterfaceSpec,
    call_akshare,
    probe_interfaces,
    to_frame,
)

AS_OF_DATE = "20260724"
TEST_SYMBOL = "600036"
OUTPUT_ROOT = LABS_DIR / "data" / "lab0"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"AKShare: {ak.__version__}")
print(f"体检参数截止日: {AS_OF_DATE}")


## Steps

### 1. 四层接口路由目录

目录保留原四本 Notebook 的主要接口，但不在一次体检中无边界调用所有接口。
`PRIMARY/BACKUP` 是新 Lab 1 的生产路由，`CATALOG` 只登记能力，
`DEFERRED/DISABLED` 不参与自动降级。


In [2]:
ROUTE_CATALOG = pd.DataFrame([
    # 行情
    ("行情", "stock_sse_summary", "上海证券交易所", "CATALOG", "市场总貌"),
    ("行情", "stock_board_industry_cons_em", "东方财富", "DISABLED", "行业成分不可依赖"),
    ("行情", "fund_etf_hist_em", "东方财富", "CATALOG", "ETF 历史行情"),
    ("行情", "stock_zh_a_hist_tx", "腾讯财经", "PRIMARY", "不复权/前复权日线"),
    ("行情", "stock_zh_a_daily", "新浪财经", "BACKUP", "行情备源与双源校验"),
    ("行情", "stock_zh_a_hist", "东方财富", "DISABLED", "当前风控面不可依赖"),
    ("行情", "mootdx.bars", "通达信 TCP", "DEFERRED", "当前实测可能返回空"),
    ("行情", "tencent_quote", "腾讯财经", "CATALOG", "指数/ETF/个股快照"),
    ("行情", "tushare.daily", "Tushare", "CATALOG", "需 Token/积分"),
    # 基础数据
    ("基础数据", "stock_info_sh_name_code", "上交所", "PRIMARY", "上交所股票清单"),
    ("基础数据", "stock_info_sz_name_code", "深交所", "PRIMARY", "深交所股票清单"),
    ("基础数据", "stock_info_bj_name_code", "北交所", "PRIMARY", "北交所股票清单"),
    ("基础数据", "mootdx.F10", "通达信 TCP", "PRIMARY", "主营业务文本验证"),
    ("基础数据", "mootdx.finance", "通达信 TCP", "CATALOG", "财务快照"),
    ("基础数据", "stock_individual_info_em", "东方财富", "CATALOG", "公司信息"),
    ("基础数据", "stock_financial_abstract", "新浪财经", "CATALOG", "财务摘要"),
    ("基础数据", "stock_financial_report_sina", "新浪财经", "CATALOG", "财务报表"),
    ("基础数据", "tencent_quote_valuation", "腾讯财经", "CATALOG", "估值快照"),
    ("基础数据", "tushare.stock_basic/fina_indicator", "Tushare", "CATALOG", "需 Token/积分"),
    ("基础数据", "stock_history_dividend_detail", "新浪财经", "PRIMARY", "已实施分红"),
    ("基础数据", "stock_fhps_detail_em", "东方财富", "BACKUP", "分红备源"),
    # 研报
    ("研报", "stock_research_report_em", "东方财富", "CATALOG", "个股研报"),
    ("研报", "eastmoney_latest_reports", "东方财富", "CATALOG", "全市场研报流"),
    ("研报", "eastmoney_industry_reports", "东方财富", "CATALOG", "行业研报"),
    ("研报", "stock_institute_recommend_detail", "新浪财经", "CATALOG", "评级记录"),
    ("研报", "ths_eps_forecast", "同花顺", "CATALOG", "一致预期 EPS"),
    ("研报", "iwencai.report_search", "iwencai", "CATALOG", "需 Key 的语义搜索"),
    ("研报", "tushare.report_rc", "Tushare", "CATALOG", "需 Token/积分"),
    # 新闻与公告
    ("新闻与公告", "stock_info_global_cls", "财联社", "CATALOG", "全市场快讯"),
    ("新闻与公告", "cls_telegraph", "财联社", "CATALOG", "签名直连快讯"),
    ("新闻与公告", "stock_info_global_em", "东方财富", "CATALOG", "7×24 快讯"),
    ("新闻与公告", "stock_news_em", "东方财富", "CATALOG", "个股新闻"),
    ("新闻与公告", "stock_notice_report", "东方财富", "CATALOG", "公告聚合"),
    ("新闻与公告", "cninfo_announcements", "巨潮资讯", "PRIMARY", "法定公告"),
    ("新闻与公告", "iwencai.news/announcement_search", "iwencai", "CATALOG", "需 Key"),
    ("新闻与公告", "mootdx.F10.latest", "通达信 TCP", "CATALOG", "公告摘要，不替代原文"),
], columns=["数据层", "接口", "真实上游", "路由决策", "用途"])

display(ROUTE_CATALOG)
display(
    ROUTE_CATALOG.groupby(["数据层", "路由决策"])
    .size().rename("接口数").reset_index()
)


,数据层,接口,真实上游,路由决策,用途
0,行情,stock_sse_summary,上海证券交易所,CATALOG,市场总貌
1,行情,stock_board_industry_cons_em,东方财富,DISABLED,行业成分不可依赖
2,行情,fund_etf_hist_em,东方财富,CATALOG,ETF 历史行情
3,行情,stock_zh_a_hist_tx,腾讯财经,PRIMARY,不复权/前复权日线
4,行情,stock_zh_a_daily,新浪财经,BACKUP,行情备源与双源校验
5,行情,stock_zh_a_hist,东方财富,DISABLED,当前风控面不可依赖
6,行情,mootdx.bars,通达信 TCP,DEFERRED,当前实测可能返回空
7,行情,tencent_quote,腾讯财经,CATALOG,指数/ETF/个股快照
8,行情,tushare.daily,Tushare,CATALOG,需 Token/积分
9,基础数据,stock_info_sh_name_code,上交所,PRIMARY,上交所股票清单


,数据层,路由决策,接口数
0,基础数据,BACKUP,1
1,基础数据,CATALOG,6
2,基础数据,PRIMARY,5
3,新闻与公告,CATALOG,7
4,新闻与公告,PRIMARY,1
5,研报,CATALOG,7
6,行情,BACKUP,1
7,行情,CATALOG,4
8,行情,DEFERRED,1
9,行情,DISABLED,2


### 2. 新 Lab 1 核心依赖连续体检


In [3]:
def fetch_tdx_f10_sample():
    client = tdx_client()
    try:
        return {"symbol": TEST_SYMBOL, "text": flatten_f10(client.F10(symbol=TEST_SYMBOL))}
    finally:
        close = getattr(client, "close", None)
        if callable(close):
            close()


def normalize_f10(value):
    frame = to_frame(value)
    frame["text_length"] = frame["text"].astype(str).str.len()
    return frame


CORE_SPECS = [
    InterfaceSpec(
        "stock_info_sh_name_code", "基础数据", "宏观", "AKShare", "上海证券交易所",
        lambda: call_akshare(ak.stock_info_sh_name_code, symbol="主板A股"),
        to_frame, ("sse.com.cn",), notes="银行股票池：上交所清单",
    ),
    InterfaceSpec(
        "stock_info_sz_name_code", "基础数据", "宏观", "AKShare", "深圳证券交易所",
        lambda: call_akshare(ak.stock_info_sz_name_code, symbol="A股列表"),
        to_frame, ("szse.cn",), notes="银行股票池：深交所清单",
    ),
    InterfaceSpec(
        "stock_info_bj_name_code", "基础数据", "宏观", "AKShare", "北京证券交易所",
        lambda: call_akshare(ak.stock_info_bj_name_code),
        to_frame, ("bse.cn",), notes="银行股票池：北交所清单",
    ),
    InterfaceSpec(
        "mootdx.F10", "基础数据", "微观", "mootdx", "通达信 TCP",
        fetch_tdx_f10_sample, normalize_f10,
        notes="当前服务器通常只返回最新提示，用主营业务文本确认行业",
    ),
    InterfaceSpec(
        "stock_zh_a_hist_tx", "行情", "微观", "AKShare", "腾讯财经",
        lambda: fetch_tencent_history(
            TEST_SYMBOL, "20260101", AS_OF_DATE, adjustment="raw"
        ),
        to_frame, ("qq.com",), notes="银行回测不复权主源",
    ),
    InterfaceSpec(
        "stock_zh_a_daily", "行情", "微观", "AKShare", "新浪财经",
        lambda: fetch_sina_history(
            TEST_SYMBOL, "20260101", AS_OF_DATE, adjustment="raw"
        ),
        to_frame, ("sina.com.cn",), notes="银行回测行情备源/校验源",
    ),
    InterfaceSpec(
        "stock_history_dividend_detail", "基础数据", "微观", "AKShare", "新浪财经",
        lambda: call_akshare(
            ak.stock_history_dividend_detail, symbol=TEST_SYMBOL, indicator="分红"
        ),
        to_frame, ("sina.com.cn",), notes="现金分红主源",
    ),
    InterfaceSpec(
        "stock_fhps_detail_em", "基础数据", "微观", "AKShare", "东方财富",
        lambda: call_akshare(ak.stock_fhps_detail_em, symbol=TEST_SYMBOL),
        to_frame, ("eastmoney.com",), notes="现金分红备源；不用于行情",
        min_interval=1.2,
    ),
]

health_registry, health_attempts, run_dir = probe_interfaces(
    CORE_SPECS, OUTPUT_ROOT, repeats=2
)
display(
    health_registry[
        ["接口", "类别", "尺度", "上游源", "状态", "成功/测试",
         "行数", "平均耗时(秒)", "错误", "测试时间"]
    ]
)


,接口,类别,尺度,上游源,状态,成功/测试,行数,平均耗时(秒),错误,测试时间
0,stock_info_sh_name_code,基础数据,宏观,上海证券交易所,AVAILABLE,2/2,1698,2.440,,2026-07-24 17:32:30
1,stock_info_sz_name_code,基础数据,宏观,深圳证券交易所,AVAILABLE,2/2,2892,1.506,,2026-07-24 17:32:33
2,stock_info_bj_name_code,基础数据,宏观,北京证券交易所,AVAILABLE,2/2,330,4.598,,2026-07-24 17:32:43
3,mootdx.F10,基础数据,微观,通达信 TCP,AVAILABLE,2/2,1,0.554,,2026-07-24 17:32:44
4,stock_zh_a_hist_tx,行情,微观,腾讯财经,AVAILABLE,2/2,134,2.022,,2026-07-24 17:32:47
5,stock_zh_a_daily,行情,微观,新浪财经,AVAILABLE,2/2,134,3.712,,2026-07-24 17:32:55
6,stock_history_dividend_detail,基础数据,微观,新浪财经,AVAILABLE,2/2,28,0.322,,2026-07-24 17:32:58
7,stock_fhps_detail_em,基础数据,微观,东方财富,AVAILABLE,2/2,26,0.530,,2026-07-24 17:33:01


### 3. 合并路由决策并保存稳定汇总

`DISABLED` 与 `DEFERRED` 是人为路由决策，不冒充接口健康状态。
东方财富分红备源仍可体检，但东方财富历史行情不会被调用。


In [4]:
decision_map = ROUTE_CATALOG.set_index("接口")["路由决策"].to_dict()
health_registry["路由决策"] = health_registry["接口"].map(decision_map).fillna("CATALOG")

manual_rows = pd.DataFrame([
    {
        "接口": "mootdx.bars",
        "类别": "行情",
        "尺度": "微观",
        "上游源": "通达信 TCP",
        "状态": "UNTESTED",
        "成功/测试": "0/0",
        "行数": 0,
        "平均耗时(秒)": 0.0,
        "错误": "",
        "测试时间": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "路由决策": "DEFERRED",
    },
    {
        "接口": "stock_zh_a_hist",
        "类别": "行情",
        "尺度": "微观",
        "上游源": "东方财富",
        "状态": "UNTESTED",
        "成功/测试": "0/0",
        "行数": 0,
        "平均耗时(秒)": 0.0,
        "错误": "",
        "测试时间": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "路由决策": "DISABLED",
    },
])
source_healthcheck = pd.concat(
    [
        health_registry[
            ["接口", "类别", "尺度", "上游源", "状态", "成功/测试",
             "行数", "平均耗时(秒)", "错误", "测试时间", "路由决策"]
        ],
        manual_rows,
    ],
    ignore_index=True,
)
stable_path = OUTPUT_ROOT / "source_healthcheck.csv"
source_healthcheck.to_csv(stable_path, index=False, encoding="utf-8-sig")
display(source_healthcheck)
print(f"稳定汇总: {stable_path.relative_to(LABS_DIR)}")
print(f"版本化快照: {run_dir.relative_to(LABS_DIR)}")


,接口,类别,尺度,上游源,状态,成功/测试,行数,平均耗时(秒),错误,测试时间,路由决策
0,stock_info_sh_name_code,基础数据,宏观,上海证券交易所,AVAILABLE,2/2,1698,2.440,,2026-07-24 17:32:30,PRIMARY
1,stock_info_sz_name_code,基础数据,宏观,深圳证券交易所,AVAILABLE,2/2,2892,1.506,,2026-07-24 17:32:33,PRIMARY
2,stock_info_bj_name_code,基础数据,宏观,北京证券交易所,AVAILABLE,2/2,330,4.598,,2026-07-24 17:32:43,PRIMARY
3,mootdx.F10,基础数据,微观,通达信 TCP,AVAILABLE,2/2,1,0.554,,2026-07-24 17:32:44,PRIMARY
4,stock_zh_a_hist_tx,行情,微观,腾讯财经,AVAILABLE,2/2,134,2.022,,2026-07-24 17:32:47,PRIMARY
5,stock_zh_a_daily,行情,微观,新浪财经,AVAILABLE,2/2,134,3.712,,2026-07-24 17:32:55,BACKUP
6,stock_history_dividend_detail,基础数据,微观,新浪财经,AVAILABLE,2/2,28,0.322,,2026-07-24 17:32:58,PRIMARY
7,stock_fhps_detail_em,基础数据,微观,东方财富,AVAILABLE,2/2,26,0.530,,2026-07-24 17:33:01,BACKUP
8,mootdx.bars,行情,微观,通达信 TCP,UNTESTED,0/0,0,0.000,,2026-07-24 17:33:01,DEFERRED
9,stock_zh_a_hist,行情,微观,东方财富,UNTESTED,0/0,0,0.000,,2026-07-24 17:33:01,DISABLED


稳定汇总: data\lab0\source_healthcheck.csv
版本化快照: data\lab0\20260724_173225


## Checks


In [5]:
allowed_status = {
    "AVAILABLE", "UNSTABLE", "BLOCKED", "BROKEN", "EMPTY", "UNTESTED"
}
assert set(source_healthcheck["状态"]).issubset(allowed_status)
assert source_healthcheck["接口"].is_unique
assert source_healthcheck.loc[
    source_healthcheck["接口"].eq("stock_zh_a_hist"), "路由决策"
].eq("DISABLED").all()
assert source_healthcheck.loc[
    source_healthcheck["接口"].eq("mootdx.bars"), "路由决策"
].eq("DEFERRED").all()
assert stable_path.is_file()
print("状态、路由决策和汇总文件检查通过")


状态、路由决策和汇总文件检查通过


## Next Steps

Lab 0 只回答“本次运行时，哪些数据入口可访问”。Lab 1 的回测还必须对每只股票执行
日期、字段、双源价格偏差、分红单位和回测范围检查。接口 `AVAILABLE` 不等于数据已经
适合回测。
